# Optimized K-Means: Seeding & Triangle Inequality AccelerationEvaluating two foundational computational and algorithmic optimizations:1. **K-Means++ Initialization (Arthur & Vassilvitskii, 2007):** Probabilistic $D^2$ seeding with an $O(\log k)$ approximation guarantee.2. **Elkan's K-Means (Elkan, 2003):** Triangle inequality bounding to skip provably redundant distance calculations.

In [ ]:
import sys
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.algorithms.traditional_kmeans import TraditionalKMeans
from src.algorithms.kmeans_plus_plus import KMeansPlusPlus
from src.algorithms.elkan_kmeans import ElkanKMeans
from src.datasets.synthetic import generate_gaussian_blobs
from src.datasets.preprocessing import preprocess_features

In [ ]:
# Multi-dimensional benchmark dataset
X_raw, _, _ = generate_gaussian_blobs(n_samples=3000, n_features=10, n_clusters=5, cluster_std=1.5, random_state=42)
X_norm, _ = preprocess_features(X_raw)

models = {
    "Traditional (Lloyd)": TraditionalKMeans(n_clusters=5, random_state=42),
    "K-Means++": KMeansPlusPlus(n_clusters=5, random_state=42),
    "Elkan K-Means": ElkanKMeans(n_clusters=5, random_state=42),
}

results = []
for name, m in models.items():
    m.fit(X_norm)
    results.append({
        "Algorithm": name,
        "Inertia": m.inertia_,
        "Iterations": m.n_iter_,
        "Runtime (s)": m.total_time_,
        "Distance Calcs": m.distance_calculations_,
    })

df_res = pd.DataFrame(results)
base_calcs = df_res.loc[0, "Distance Calcs"]
df_res["Distance Reduction (%)"] = ((base_calcs - df_res["Distance Calcs"]) / base_calcs) * 100.0
df_res

In [ ]:
# Distance calculation reduction and runtime comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2), dpi=150)
x = range(len(df_res))

ax1.bar(x, df_res["Distance Calcs"], color=["#4C72B0", "#55A868", "#C44E52"], width=0.5, edgecolor="#333333")
ax1.set_title("Total Distance Evaluations", fontsize=10.5, fontweight="bold")
ax1.set_xticks(x)
ax1.set_xticklabels(df_res["Algorithm"], rotation=15, ha="right", fontsize=8.5)
ax1.grid(axis="y", linestyle="--", alpha=0.5)

ax2.bar(x, df_res["Runtime (s)"], color=["#4C72B0", "#55A868", "#C44E52"], width=0.5, edgecolor="#333333")
ax2.set_title("Runtime (seconds)", fontsize=10.5, fontweight="bold")
ax2.set_xticks(x)
ax2.set_xticklabels(df_res["Algorithm"], rotation=15, ha="right", fontsize=8.5)
ax2.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()